In [0]:
!pip install pypdf

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 6.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
!pip uninstall --yes fitz
!pip install pymupdf

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 70.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./AID_component_develop/Notebook/other/DataBricksSetting/BlobDatabricksSettings

In [1]:
import requests
import time
import glob
import json
import re
import pypdf
import os
import fitz
import mimetypes
from dataclasses import dataclass
from PIL import Image
from mimetypes import guess_type
import json

In [0]:
@dataclass
class DocumentIntelligenceClient:
    endpoint: str 
    key: str 
    api_version: str 
    custom_flag:bool = False

    def _request(self, file_url: str, model_id, pages=1) -> str:
        url = f"{self.endpoint}documentintelligence/documentModels/{model_id}:analyze?api-version={self.api_version}&pages={pages}"
        headers = {"Ocp-Apim-Subscription-Key": self.key}
        response = requests.post(url, headers=headers, json={"urlSource": file_url})
        return response.headers["apim-request-id"]
    
    def _get_result(self, request_id, model_id,format_name) -> dict:
        url = f"{self.endpoint}documentintelligence/documentModels/" \
              + f"{model_id}/analyzeResults/{request_id}?api-version={self.api_version}&outputContentFormat={format_name}"
        headers = {"Ocp-Apim-Subscription-Key": self.key}
        return requests.get(url, headers=headers).json()

    # 段落番号を取得するための関数
    def _get_paragraph_numbers(self, elements_data):
        paragraph_numbers = []
        if elements_data.get('elements'):
            for paragraph in elements_data['elements']:
                paragraph_numbers.append(paragraph.split('/')[-1])  # '/paragraphs/4' のような形式なので、最後の番号を取得
        return paragraph_numbers

    # 段落番号に対応するContentを結合するための関数
    def _merge_paragraphs(self, paragraph_numbers, content_data):
        merged_content = []
        if paragraph_numbers:
            for paragraph_num in paragraph_numbers:
                # 段落番号が一致するContentを検索
                for index, content in enumerate(content_data):
                    if f'/paragraphs/{index + 1}' == f'/paragraphs/{paragraph_num}':
                        merged_content.append(content['content'])
        return ' '.join(merged_content)

    def _ocr_to_markdown(self, table):
        # 元の方法でテーブルをマークダウン形式に変換
        caption = table.get('caption', {}).get('content', 'Table')
        markdown_table = f"## {caption}\n\n"

        rows = {}
        max_col_index = 0
        for cell in table.get('cells', []):
            row_index = cell['rowIndex']
            col_index = cell['columnIndex']
            col_span = cell.get('columnSpan', 1)
            content = cell.get('content', '')

            if row_index not in rows:
                rows[row_index] = {}
            rows[row_index][col_index] = {'content': content, 'col_span': col_span}
            max_col_index = max(max_col_index, col_index + col_span - 1)

        header_row = []
        separator_row = []
        for col_index in range(max_col_index + 1):
            cell = rows.get(0, {}).get(col_index, {})
            content = cell.get('content', '')
            col_span = cell.get('col_span', 1)
            header_row.append(content)
            separator_row.append('---' * col_span)

        markdown_table += '| ' + ' | '.join(header_row) + ' |\n'
        markdown_table += '| ' + ' | '.join(separator_row) + ' |\n'

        for row_index in range(1, len(rows)):
            row_content = []
            for col_index in range(max_col_index + 1):
                cell = rows.get(row_index, {}).get(col_index, {})
                content = cell.get('content', '')
                col_span = cell.get('col_span', 1)
                row_content.append(content)
                if col_span > 1:
                    row_content.extend([''] * (col_span - 1))

            markdown_table += '| ' + ' | '.join(row_content) + ' |\n'

        return markdown_table

    
    def get_raw_result(self, url, retry_count=2, waiting_time=5, model="prebuilt-read"):  
        request_id = self._request(url, model)
        for count in range(retry_count):
            time.sleep(waiting_time)
            result = self._get_result(request_id, model)
            if result.get("status", "failed") == "succeeded":
                break
        return result

    def _to_output_format(self, result: dict) -> [dict]:
        if self.custom_flag:
            annlyze_result = self.fields_analyze(result)
        else:
            if result["analyzeResult"]["content"]:
                annlyze_result = result["analyzeResult"]["content"].replace("\"","").replace("\'","")
            else:
                annlyze_result = result["analyzeResult"]["content"]
        pages = result["analyzeResult"]["pages"]
        lines_text = [{
            "page_number": page["pageNumber"],
            "content": " ".join([line.get("content") for line in page["lines"]]),
        } for page in pages]
        for text in lines_text:
            if text.get("content"):
                text["content"].replace("\"","").replace("\'","")
        return (annlyze_result,lines_text)

    def get_raw_result_from_binary(self, file_content, format_name ,retry_count=100, waiting_time=5):
        if self.custom_flag:
            model =  format_name
        else:
            model = 'prebuilt-layout'
        url = f"{self.endpoint}documentintelligence/documentModels/{model}:analyze?api-version={self.api_version}&outputContentFormat={format_name}"
        headers = {
            "Content-Type": "application/pdf",
            "Ocp-Apim-Subscription-Key": self.key
        }

        response = requests.post(url=url, headers=headers, data=file_content)   
        request_id = response.headers["apim-request-id"]

        for count in range(retry_count):
            result = self._get_result(request_id, model,format_name)
            if result.get("status", "failed") == "succeeded":
                return result
                # return self._to_output_format(result)
            time.sleep(waiting_time)
        return result
    
    def get_ocr_result(self, file_path, format_name='text'):
        with open(file_path, "rb") as f:
            file_content = f.read()
        return self.get_raw_result_from_binary(file_content,format_name)
    
    def fields_analyze(self, result:dict):
        fields_string = dict(filter(lambda x: x[1]['type']=='string',result["analyzeResult"]["documents"][0]["fields"].items()))
        fields_string_list=[]
        for field in fields_string:
            if fields_string[field].get("content"):
                fields_string_list.append((field, fields_string[field].get("content").replace("\"","").replace("\'","")))
            else:
                fields_string_list.append((field, fields_string[field].get("content")))
        return dict(fields_string_list)
    
    def split_and_merge_pdf(self, input_file_path,split_file_dir,chunk_size):
        file_name = os.path.basename(input_file_path)
        file_name_without_extension = os.path.splitext(file_name)[0]
        pdf_len = len(pypdf.PdfReader(input_file_path).pages)
        if pdf_len > 1800:
            os.makedirs(f"{split_file_dir}/{file_name_without_extension}",exist_ok=True)
            for i in range(0, pdf_len, chunk_size):
                merger = pypdf.PdfMerger()
                merger.append(input_file_path, pages=pypdf.PageRange(f'{i}:{i + chunk_size}'))
                merger.write(f"{split_file_dir}/{file_name_without_extension}/{file_name_without_extension}_from_{i+1}_to_{i + chunk_size}.pdf")
                merger.close()
            merged_result = []
            split_file_list = glob.glob(os.path.join(f"{split_file_dir}/{file_name_without_extension}", '*'))
            for split_file_path in split_file_list:
                split_file_result = self.get_ocr_result(split_file_path)
                pattern = r'_from_(\d+)_to_\d+\.pdf'
                from_number = re.search(pattern, split_file_path).group(1)
                diff = int(from_number) - 1
                for r in split_file_result:
                    r["page_number"] = r["page_number"] + diff
                    merged_result.append(r)
            return merged_result
        else:
            return self.get_ocr_result(input_file_path)

    def merge_bounding_regions(self, bounding_regions):  
        if not bounding_regions:
            return []  
    
        min_x = float('inf')  
        min_y = float('inf')  
        max_x = float('-inf')  
        max_y = float('-inf')  
    
        for region in bounding_regions:  
            polygon = region["polygon"]  
            for i in range(0, len(polygon), 2):  
                x = polygon[i]  
                y = polygon[i + 1]  
                if x < min_x:  
                    min_x = x  
                if y < min_y:  
                    min_y = y  
                if x > max_x:  
                    max_x = x  
                if y > max_y:  
                    max_y = y  
    
        merged_polygon = [  
            min_x, min_y,  # 左上  
            max_x, min_y,  # 右上  
            max_x, max_y,  # 右下  
            min_x, max_y   # 左下  
        ]  
    
        return merged_polygon
    
    def is_within_table_or_figure(self, paragraph, regions):
        px1, py1, px2, py2, px3, py3, px4, py4 = paragraph["boundingRegions"][0]["polygon"]  
        for region in regions:
            rx1, ry1, rx2, ry2, rx3, ry3, rx4, ry4 = region
            if (rx1 <= px1 <= rx2 and ry1 <= py1 <= ry3) or (rx1 <= px3 <= rx2 and ry1 <= py3 <= ry3):
                return True
        return False
    
    def is_within_exclusion_zone(self, bounding_regions, exclusion_zones):  
        for bounding_region in bounding_regions:  
            polygon = bounding_region.get("polygon", [])  
            if not isinstance(polygon, list):  
                continue
    
            px1, py1, px2, py2, px3, py3, px4, py4 = polygon  
    
            for zone in exclusion_zones:
                zx1, zy1, zx2, zy2, zx3, zy3, zx4, zy4 = zone  
                if (px1 <= zx1 <= px2 and py1 <= zy1 <= py3) or (px1 <= zx3 <= px2 and py1 <= zy3 <= py3):  
                    return True  
    
        return False
    
    def _concatenate_paragraphs(self, paragraphs):
        return ' '.join(paragraph["content"] for paragraph in paragraphs)
     
    def extract_content_by_page(self, extract_keyword, file_path, file_name, result: dict) -> list:  
        analyze_result = result["analyzeResult"]
        pages = analyze_result.get("pages", [])  
        paragraphs = analyze_result.get("paragraphs", [])  
        tables = analyze_result.get("tables", [])  
        figures = analyze_result.get("figures", [])  
        extracted_pages = []  
        caution_keywords = extract_keyword

        id = 1
        
        for page in pages:  
            page_number = page["pageNumber"]
            print(f"{page_number}ページ処理開始")
            
            content_by_type = []  
            elements = []
            exclusion_zones = []

            # 除外ゾーンの設定
            for paragraph in paragraphs:
                if paragraph["boundingRegions"][0]["pageNumber"] == page_number:
                    if paragraph["content"] == "Infineon":
                        exclusion_zones.append(paragraph["boundingRegions"][0]["polygon"])

            # 各要素の抽出
            for paragraph in paragraphs:  
                if paragraph["boundingRegions"][0]["pageNumber"] == page_number:
                    if paragraph["content"] == "Infineon":
                        continue
                    elements.append((paragraph, "paragraph"))  
            
            for table in tables:  
                if table["boundingRegions"][0]["pageNumber"] == page_number:  
                    elements.append((table, "table"))  
            
            for figure in figures:  
                if figure["boundingRegions"][0]["pageNumber"] == page_number:
                    if self.is_within_exclusion_zone(figure["boundingRegions"], exclusion_zones):
                        continue
                    elements.append((figure, "figure"))  
            
            # 上下位置でソート
            elements.sort(key=lambda x: x[0]["boundingRegions"][0]["polygon"][1])  
            
            # 図・表の領域リスト
            table_figure_regions = []  
            
            # 各要素の分類と処理
            for element, element_type in elements:  
                if element.get("role") in ["pageHeader", "pageFooter", "pageNumber"]:  
                    continue

                content = element.get("content", "")
                classified_type = "その他の文章"

                if element_type == "table":  
                    content = self._ocr_to_markdown(element)  # テーブルをマークダウン形式に変換
                    merged_polygon = self.merge_bounding_regions(element["boundingRegions"])
                    table_figure_regions.append(merged_polygon)
                    classified_type = "表"
                    
                elif element_type == "figure":  
                    # 図の中の段落を結合して一つの塊にする
                    # figure_paragraphs = [
                    #     paragraph for paragraph in paragraphs
                    #     if self.is_within_table_or_figure(paragraph, [self.merge_bounding_regions(element["boundingRegions"])])
                    # ]
                    paragraph_numbers = self._get_paragraph_numbers(element)
                    content = self._merge_paragraphs(paragraph_numbers, analyze_result["paragraphs"])
                    merged_polygon = self.merge_bounding_regions(element["boundingRegions"])
                    table_figure_regions.append(merged_polygon)
                    classified_type = "図"
                    
                elif any(keyword in content.lower() for keyword in caution_keywords):
                    if self.is_within_table_or_figure(element, table_figure_regions):  
                        classified_type = "図・表内の注意点"  
                    else:  
                        classified_type = "図・表外の注意点"
                        
                elif element.get("role") == "sectionHeading":  
                    classified_type = "章タイトル"  

                content_by_type.append({
                    "type": classified_type,  
                    "content": content,  
                    "boundingRegions": element.get("boundingRegions", [])  
                })
            
            # "その他の文章" で図・表内に含まれるものをフィルタリング
            filtered_content_by_type = []
            for p in content_by_type:
                if p["type"] == "その他の文章" and self.is_within_table_or_figure(p, table_figure_regions):
                    continue  
                filtered_content_by_type.append(p)

            # フィルタリング後の内容を処理
            for p in filtered_content_by_type:
                merged_region = self.merge_bounding_regions(p["boundingRegions"])
                figure_table_type = None  
                if p["type"] == "表":  
                    figure_table_type = 1  
                elif p["type"] == "図":
                    figure_table_type = 0
                
                yield {
                    "id": id,
                    "file_name": file_name,
                    "file_path": str(file_path),
                    "page_number": page_number,
                    "type": p["type"],
                    "figure_table_type": figure_table_type,
                    "content": p["content"],  
                    "region": merged_region
                }

                id += 1


    def get_processed_result(self, file_path, split_file_path, extract_file_path, chunk_size, extract_keyword, file_name, format_name='', page_range=None):  
        total_pages = len(pypdf.PdfReader(file_path).pages)
        
        if page_range is not None:  
            start_page, end_page = page_range
            print(f"全ページ数: {total_pages}, start_page: {start_page}, end_page: {end_page}")

            if start_page < 1 or end_page > total_pages:  
                raise ValueError("ページ数を超過しています")

            base_name = os.path.splitext(file_name)[0]  
            extract_file_name = f"{base_name}_{start_page}_to_{end_page}_extracted.pdf"
            extract_file_full_path = os.path.join(extract_file_path, extract_file_name)

            input_file_path = extract_file_full_path
        else:  
            input_file_path = file_path

        print("OCR開始")
        processed_result = self.split_and_merge_pdf(input_file_path, split_file_path, chunk_size)
        print("OCR終了")
        return list(self.extract_content_by_page(extract_keyword, file_path, file_name, processed_result))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.0/388.0 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-80a47f6e-df32-4b2e-b569-3dfcf1054a9c
    Can't uninstall 'typing_extensions'. No files were found to uninstall.



[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [0]:
import pprint
di = DocumentIntelligenceClient(endpoint="https://20240525test.cognitiveservices.azure.com/",key="242dc7331a484ecdae1f15ec90c957a8",
                                api_version="2024-07-31-preview")

di.custom_flag = False

filepath = "/dbfs/mnt/rag-poc-q2-container/pdfs/ノウハウ_機密事項あり取り扱い注意/ゴム/NRU007_ゴムの加工.pdf"

split_f_path = "/dbfs/mnt/rag-poc-q2-container/pdfs/splits/"
extract_f_path = "/dbfs/mnt/rag-poc-q2-container/pdfs/extracts/"
chunk_size = 1000
extract_keywords = []
new_file_name = "new-file-name01"

result = di.get_processed_result(filepath, split_f_path, extract_f_path, chunk_size, extract_keywords, new_file_name)

pprint.pprint(result)

OCR開始
OCR終了
1ページ処理開始
<class 'dict'>
<class 'dict'>
<class 'dict'>
2ページ処理開始
<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'content': '▶ Eralit',
  'figure_table_type': None,
  'file_name': 'new-file-name01',
  'file_path': '/dbfs/mnt/rag-poc-q2-container/pdfs/ノウハウ_機密事項あり取り扱い注意/ゴム/NRU007_ゴムの加工.pdf',
  'id': 1,
  'page_number': 1,
  'region': [2.9022, 1.2058, 3.1629, 1.2058, 3.1629, 1.276, 2.9022, 1.276],
  'type': 'その他の文章'},
 {'content': 'ゴムの加工工程を簡単に示すと、図1のようになる。各種原料が計量された後、配合·混練の混練工程を経て、成形·加硫の '
             '成形工程でゴム製品が出来上がるのである。',
  'figure_table_type': None,
  'file_name': 'new-file-name01',
  'file_path': '/dbfs/mnt/rag-poc-q2-container/pdfs/ノウハウ_機密事項あり取り扱い注意/ゴム/NRU007_ゴムの加工.pdf',
  'id': 2,
  'page_number': 1,
  'region': [0.7601, 1.3254, 3.2791, 1.3254, 3.2791, 1.461, 0.7601, 1.461],
  'type': 'その他の文章'},
 {'content': 'ゴムの加工工程を簡単に示すと、図1のようになる。各種原料が計量された後、配合·混練の混練工程を経て、成形·加硫の '
             '成形工程でゴム製品が出来上がるのである。 原料メーカー ゴム加エメーカーで実施 原料ゴム 混練工程 成形工程 已合藥品 配合 '
             '加確 ゴム製品 加工溫度:

In [0]:
#マウント処理
STORAGE_ACCOUNT_NAME = "tmcgenaistrage"
BLOB_CONTAINER_NAMES = [
    "rag-poc-q2-container"
]
STORAGE_ACCOUNT_KEY = "s2o2JlrqYlSByT+K0pHUQSE8JnwRo+tvIlRSkfas0yHzhhXjat/5ZZyEi8knQ3CvPhjIj49aTF5E+AStEi29EA=="

bds = BlobDatabricksSettings(STORAGE_ACCOUNT_NAME, STORAGE_ACCOUNT_KEY)

for blob_container_name in BLOB_CONTAINER_NAMES:
    bds.mount_blob(blob_container_name, f"/mnt/{blob_container_name}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-1896103647814682>, line 2
      1 import pprint
----> 2 di = DocumentIntelligenceClient(endpoint="https://20240525test.cognitiveservices.azure.com/",key="242dc7331a484ecdae1f15ec90c957a8",
      3                                 api_version="2024-07-31-preview")
      5 di.custom_flag = False
      7 filepath = "/dbfs/mnt/rag-poc-q2-container/pdfs/ノウハウ_機密事項あり取り扱い注意/ゴム/NRU007_ゴムの加工.pdf"

NameError: name 'DocumentIntelligenceClient' is not defined

In [0]:
# @dataclass
# class DocumentIntelligenceClient:
#     endpoint: str 
#     key: str 
#     api_version: str 
#     custom_flag:bool = False

#     def _request(self, file_url: str, model_id, pages=1) -> str:
#         url = f"{self.endpoint}documentintelligence/documentModels/{model_id}:analyze?api-version={self.api_version}&pages={pages}"
#         headers = {"Ocp-Apim-Subscription-Key": self.key}
#         response = requests.post(url, headers=headers, json={"urlSource": file_url})
#         return response.headers["apim-request-id"]
    
#     def _get_result(self, request_id, model_id) -> dict:
#         url = f"{self.endpoint}documentintelligence/documentModels/" \
#               + f"{model_id}/analyzeResults/{request_id}?api-version={self.api_version}"
#         headers = {"Ocp-Apim-Subscription-Key": self.key}
#         return requests.get(url, headers=headers).json()
    
#     import json

#     def _ocr_to_markdown(self,ocr_data):
#         markdown_tables = []
        
#         for table in ocr_data:
#             # Extracting caption for the table
#             caption = table.get('caption', {}).get('content', 'Table')
#             markdown_table = f"## {caption}\n\n"

#             # Group cells by rows
#             rows = {}
#             max_col_index = 0
#             for cell in table.get('cells', []):
#                 row_index = cell['rowIndex']
#                 col_index = cell['columnIndex']
#                 col_span = cell.get('columnSpan', 1)
#                 content = cell.get('content', '')

#                 if row_index not in rows:
#                     rows[row_index] = {}
#                 rows[row_index][col_index] = {'content': content, 'col_span': col_span}
#                 max_col_index = max(max_col_index, col_index + col_span - 1)

#             # Constructing markdown table header and separator
#             header_row = []
#             separator_row = []
#             for col_index in range(max_col_index + 1):
#                 cell = rows.get(0, {}).get(col_index, {})
#                 content = cell.get('content', '')
#                 col_span = cell.get('col_span', 1)
#                 header_row.append(content)
#                 separator_row.append('---' * col_span)

#             markdown_table += '| ' + ' | '.join(header_row) + ' |\n'
#             markdown_table += '| ' + ' | '.join(separator_row) + ' |\n'

#             # Constructing markdown table rows
#             for row_index in range(1, len(rows)):
#                 row_content = []
#                 for col_index in range(max_col_index + 1):
#                     cell = rows.get(row_index, {}).get(col_index, {})
#                     content = cell.get('content', '')
#                     col_span = cell.get('col_span', 1)
#                     row_content.append(content)
#                     if col_span > 1:
#                         row_content.extend([''] * (col_span - 1))

#                 markdown_table += '| ' + ' | '.join(row_content) + ' |\n'

#             markdown_tables.append(markdown_table)

#         return "\n".join(markdown_tables)

    
#     def get_raw_result(self, url, retry_count=2, waiting_time=5, model="prebuilt-read"):  
#         request_id = self._request(url, model)
#         for count in range(retry_count):
#             time.sleep(waiting_time)
#             result = self._get_result(request_id, model)
#             if result.get("status", "failed") == "succeeded":
#                 break
#         return result

#     def _to_output_format(self, result: dict) -> [dict]:
#         if self.custom_flag:
#             annlyze_result = self.fields_analyze(result)
#         else:
#             if result["analyzeResult"]["content"]:
#                 annlyze_result = result["analyzeResult"]["content"].replace("\"","").replace("\'","")
#             else:
#                 annlyze_result = result["analyzeResult"]["content"]
#         pages = result["analyzeResult"]["pages"]
#         lines_text = [{
#             "page_number": page["pageNumber"],
#             "content": " ".join([line.get("content") for line in page["lines"]]),
#         } for page in pages]
#         for text in lines_text:
#             if text.get("content"):
#                 text["content"].replace("\"","").replace("\'","")
#         return (annlyze_result,lines_text)

#     def get_raw_result_from_binary(self, file_content, format_name ,retry_count=100, waiting_time=5):
#         if self.custom_flag:
#             model =  format_name
#         else:
#             model = 'prebuilt-layout'
#         url = f"{self.endpoint}documentintelligence/documentModels/{model}:analyze?api-version={self.api_version}"
#         headers = {
#             "Content-Type": "application/pdf",
#             "Ocp-Apim-Subscription-Key": self.key
#         }

#         response = requests.post(url=url, headers=headers, data=file_content)    
#         request_id = response.headers["apim-request-id"]

#         for count in range(retry_count):
#             result = self._get_result(request_id, model)
#             if result.get("status", "failed") == "succeeded":
#                 return result
#                 # return self._to_output_format(result)
#             time.sleep(waiting_time)
#         return result
    
#     def get_ocr_result(self, file_path, format_name=''):
#         with open(file_path, "rb") as f:
#             file_content = f.read()
#         return self.get_raw_result_from_binary(file_content,format_name)
    
#     def fields_analyze(self, result:dict):
#         fields_string = dict(filter(lambda x: x[1]['type']=='string',result["analyzeResult"]["documents"][0]["fields"].items()))
#         fields_string_list=[]
#         for field in fields_string:
#             if fields_string[field].get("content"):
#                 fields_string_list.append((field, fields_string[field].get("content").replace("\"","").replace("\'","")))
#             else:
#                 fields_string_list.append((field, fields_string[field].get("content")))
#         return dict(fields_string_list)
    
#     # def split_and_merge_pdf(self, input_file_path,split_file_dir,chunk_size):
#     #     file_name = os.path.basename(input_file_path)
#     #     file_name_without_extension = os.path.splitext(file_name)[0]
#     #     pdf_len = len(pypdf.PdfReader(input_file_path).pages)
#     #     if pdf_len > 1800:
#     #         os.makedirs(f"{split_file_dir}/{file_name_without_extension}",exist_ok=True)
#     #         for i in range(0, pdf_len, chunk_size):
#     #             merger = pypdf.PdfMerger()
#     #             merger.append(input_file_path, pages=pypdf.PageRange(f'{i}:{i + chunk_size}'))
#     #             merger.write(f"{split_file_dir}/{file_name_without_extension}/{file_name_without_extension}_from_{i+1}_to_{i + chunk_size}.pdf")
#     #             merger.close()
#     #         merged_result = []
#     #         split_file_list = glob.glob(os.path.join(f"{split_file_dir}/{file_name_without_extension}", '*'))
#     #         for split_file_path in split_file_list:
#     #             split_file_result = self.get_ocr_result(split_file_path)[1]
#     #             pattern = r'_from_(\d+)_to_\d+\.pdf'
#     #             from_number = re.search(pattern, split_file_path).group(1)
#     #             diff = int(from_number) - 1
#     #             for r in split_file_result:
#     #                 r["page_number"] = r["page_number"] + diff
#     #                 merged_result.append(r)
#     #         return merged_result
#     #     else:
#     #         return self.get_ocr_result(input_file_path)[1]

#     def split_and_merge_pdf(self, input_file_path,split_file_dir,chunk_size):
#         file_name = os.path.basename(input_file_path)
#         file_name_without_extension = os.path.splitext(file_name)[0]
#         pdf_len = len(pypdf.PdfReader(input_file_path).pages)
#         if pdf_len > 1800:
#             os.makedirs(f"{split_file_dir}/{file_name_without_extension}",exist_ok=True)
#             for i in range(0, pdf_len, chunk_size):
#                 merger = pypdf.PdfMerger()
#                 merger.append(input_file_path, pages=pypdf.PageRange(f'{i}:{i + chunk_size}'))
#                 merger.write(f"{split_file_dir}/{file_name_without_extension}/{file_name_without_extension}_from_{i+1}_to_{i + chunk_size}.pdf")
#                 merger.close()
#             merged_result = []
#             split_file_list = glob.glob(os.path.join(f"{split_file_dir}/{file_name_without_extension}", '*'))
#             for split_file_path in split_file_list:
#                 split_file_result = self.get_ocr_result(split_file_path)
#                 pattern = r'_from_(\d+)_to_\d+\.pdf'
#                 from_number = re.search(pattern, split_file_path).group(1)
#                 diff = int(from_number) - 1
#                 for r in split_file_result:
#                     r["page_number"] = r["page_number"] + diff
#                     merged_result.append(r)
#             return merged_result
#         else:
#             return self.get_ocr_result(input_file_path)

#     def merge_bounding_regions(self, bounding_regions):  
#         if not bounding_regions:
#             return []  
    
#         min_x = float('inf')  
#         min_y = float('inf')  
#         max_x = float('-inf')  
#         max_y = float('-inf')  
    
#         for region in bounding_regions:  
#             polygon = region["polygon"]  
#             for i in range(0, len(polygon), 2):  
#                 x = polygon[i]  
#                 y = polygon[i + 1]  
#                 if x < min_x:  
#                     min_x = x  
#                 if y < min_y:  
#                     min_y = y  
#                 if x > max_x:  
#                     max_x = x  
#                 if y > max_y:  
#                     max_y = y  
    
#         merged_polygon = [  
#             min_x, min_y,  # 左上  
#             max_x, min_y,  # 右上  
#             max_x, max_y,  # 右下  
#             min_x, max_y   # 左下  
#         ]  
    
#         return merged_polygon
    
#     def is_within_table_or_figure(self, paragraph, regions):
#         px1, py1, px2, py2, px3, py3, px4, py4 = paragraph["boundingRegions"][0]["polygon"]  
#         for region in regions:
#             rx1, ry1, rx2, ry2, rx3, ry3, rx4, ry4 = region
#             if (rx1 <= px1 <= rx2 and ry1 <= py1 <= ry3) or (rx1 <= px3 <= rx2 and ry1 <= py3 <= ry3):
#                 return True
#         return False
    
#     def is_within_exclusion_zone(self, bounding_regions, exclusion_zones):  
#         for bounding_region in bounding_regions:  
#             polygon = bounding_region.get("polygon", [])  
#             if not isinstance(polygon, list):  
#                 continue
    
#             px1, py1, px2, py2, px3, py3, px4, py4 = polygon  
    
#             for zone in exclusion_zones:
#                 zx1, zy1, zx2, zy2, zx3, zy3, zx4, zy4 = zone  
#                 if (px1 <= zx1 <= px2 and py1 <= zy1 <= py3) or (px1 <= zx3 <= px2 and py1 <= zy3 <= py3):  
#                     return True  
    
#         return False
            
#     def extract_content_by_page(self, extract_keyword, file_path, file_name, result: dict) -> list:  
#         analyze_result = result["analyzeResult"]
#         pages = analyze_result.get("pages", [])  
#         paragraphs = analyze_result.get("paragraphs", [])  
#         tables = analyze_result.get("tables", [])  
#         figures = analyze_result.get("figures", [])  
#         extracted_pages = []  
#         caution_keywords = extract_keyword

#         id = 1
        
#         for page in pages:  
#             page_number = page["pageNumber"]
#             print(f"{page_number}ページ処理開始")
            
#             content_by_type = []  
#             elements = []
#             exclusion_zones = []

#             # 除外ゾーンの設定
#             for paragraph in paragraphs:
#                 if paragraph["boundingRegions"][0]["pageNumber"] == page_number:
#                     if paragraph["content"] == "Infineon":
#                         exclusion_zones.append(paragraph["boundingRegions"][0]["polygon"])

#             # 各要素の抽出
#             for paragraph in paragraphs:  
#                 if paragraph["boundingRegions"][0]["pageNumber"] == page_number:
#                     if paragraph["content"] == "Infineon":
#                         continue
#                     elements.append((paragraph, "paragraph"))  
        
#             for table in tables:  
#                 if table["boundingRegions"][0]["pageNumber"] == page_number:  
#                     elements.append((table, "table"))  
        
#             for figure in figures:  
#                 if figure["boundingRegions"][0]["pageNumber"] == page_number:
#                     if self.is_within_exclusion_zone(figure["boundingRegions"], exclusion_zones):
#                         continue
#                     elements.append((figure, "figure"))  
        
#             # 上下位置でソート
#             elements.sort(key=lambda x: x[0]["boundingRegions"][0]["polygon"][1])  
        
#             # 図・表の領域リスト
#             table_figure_regions = []  
        
#             # 各要素の分類と処理
#             for element, element_type in elements:  
#                 if element.get("role") in ["pageHeader", "pageFooter", "pageNumber"]:  
#                     continue
        
#                 content = element.get("content", "")
#                 classified_type = "その他の文章"
        
#                 if element_type == "table":  
#                     content = element.get("content", content)
#                     merged_polygon = self.merge_bounding_regions(element["boundingRegions"])
#                     table_figure_regions.append(merged_polygon)
#                     classified_type = "表"
                    
#                 elif element_type == "figure":  
#                     content = element.get("caption", {}).get("content", content)
#                     merged_polygon = self.merge_bounding_regions(element["boundingRegions"])
#                     table_figure_regions.append(merged_polygon)
#                     classified_type = "図"
                    
#                 elif any(keyword in content.lower() for keyword in caution_keywords):
#                     if self.is_within_table_or_figure(element, table_figure_regions):  
#                         classified_type = "図・表内の注意点"  
#                     else:  
#                         classified_type = "図・表外の注意点"
                        
#                 elif element.get("role") == "sectionHeading":  
#                     classified_type = "章タイトル"  
        
#                 content_by_type.append({
#                     "type": classified_type,  
#                     "content": content,  
#                     "boundingRegions": element.get("boundingRegions", [])  
#                 })
        
#             # "その他の文章" で図・表内に含まれるものをフィルタリング
#             filtered_content_by_type = []
#             for p in content_by_type:
#                 if p["type"] == "その他の文章" and self.is_within_table_or_figure(p, table_figure_regions):
#                     continue  
#                 filtered_content_by_type.append(p)

#             # フィルタリング後の内容を処理
#             for p in filtered_content_by_type:
#                 merged_region = self.merge_bounding_regions(p["boundingRegions"])
#                 figure_table_type = None  
#                 if p["type"] == "表":  
#                     figure_table_type = 1  
#                 elif p["type"] == "図":
#                     figure_table_type = 0
                
#                 yield {
#                     "id": id,
#                     "file_name": file_name,
#                     "file_path": str(file_path),
#                     "page_number": page_number,
#                     "type": p["type"],
#                     "figure_table_type": figure_table_type,
#                     "content": p["content"],
#                     "region": merged_region
#                 }

#                 id += 1

    
#     def extract_pages(self, input_file_path, output_file_path, start_page, end_page):  
#             reader = pypdf.PdfReader(input_file_path)
#             writer = pypdf.PdfWriter()  
    
#             for page_num in range(start_page - 1, end_page):  
#                 writer.add_page(reader.pages[page_num])  
    
#             with open(output_file_path, 'wb') as output_file:  
#                 writer.write(output_file) 

#     def get_processed_result(self, file_path, split_file_path, extract_file_path, chunk_size, extract_keyword, file_name, format_name='', page_range=None):  
#         # raw_result = self.get_ocr_result(file_path, format_name)
#         # analyze_result = raw_result["analyzeResult"]  
#         # total_pages = len(analyze_result.get("pages", []))
#         total_pages = len(pypdf.PdfReader(file_path).pages)
        
#         if page_range is not None:  
#             start_page, end_page = page_range
#             print(f"全ページ数: {total_pages}, start_page: {start_page}, end_page: {end_page}")

#             if start_page < 1 or end_page > total_pages:  
#                 raise ValueError("ページ数を超過しています")

#             base_name = os.path.splitext(file_name)[0]  
#             extract_file_name = f"{base_name}_{start_page}_to_{end_page}_extracted.pdf"
#             extract_file_full_path = os.path.join(extract_file_path, extract_file_name)

#             self.extract_pages(file_path, extract_file_full_path, start_page, end_page)  
#             input_file_path = extract_file_full_path
#         else:  
#             input_file_path = file_path

#         print("OCR開始")
#         processed_result = self.split_and_merge_pdf(input_file_path, split_file_path, chunk_size)
#         print("Result:"+str(processed_result))
#         print("OCR終了")
#         return list(self.extract_content_by_page(extract_keyword, file_path, file_name, processed_result))
        
#     # 図取得
#     def get_figure(self, result, file_name):
#         result_list = []
#         page_number = 0
#         captionBR = []
#         polygon = []
#         total = 0
#         item_id = ""
#         count = 1
#         subcount = 1

#         if result["figures"]:
#             total = len(result["figures"])
#             for i, figures in enumerate(result["figures"]):
#                 if 'caption' in figures:
#                     caption_boundingRegions = figures['caption'].get("boundingRegions")  
#                     if caption_boundingRegions:  
#                         for item in caption_boundingRegions:  
#                             captionBR = item.get('polygon')

#                 boundingRegions = figures.get("boundingRegions")
#                 if boundingRegions:

#                     # # タイトル含有用
#                     # min_x = float('inf')  
#                     # min_y = float('inf')  
#                     # max_x = float('-inf')  
#                     # max_y = float('-inf')  
#                     # # タイトル含有用

#                     for j, item in enumerate(boundingRegions):
#                         if captionBR != item.get('polygon'):
#                             page_number = item.get('pageNumber')
#                             polygon = item.get('polygon')

#                             # # タイトル含有用
#                             # # 各ポリゴンの座標をチェックして最小/最大値を更新  
#                             # for k in range(0, len(polygon), 2):  
#                             #     x = polygon[k]  
#                             #     y = polygon[k+1]  
#                             #     if x < min_x:  
#                             #         min_x = x  
#                             #     if y < min_y:  
#                             #         min_y = y  
#                             #     if x > max_x:  
#                             #         max_x = x  
#                             #     if y > max_y:  
#                             #         max_y = y   
        
#                             # # 最小値と最大値から新しいバウンディングボックスを作成  
#                             # combined_polygon = [  
#                             #     min_x, min_y,  
#                             #     max_x, min_y,  
#                             #     max_x, max_y,  
#                             #     min_x, max_y  
#                             # ]
#                             # # タイトル含有用

#                             item_id = f"{i+1}-{subcount}"
#                             subcount += 1
#                             item_id = f"{i+1}"
        
#                         # # タイトル含有用
#                         # figure_info = {"id": i+1, "file_name": file_name, "type": 0, "page_number": page_number, "itemId": item_id, "polygon": combined_polygon}
#                         # # タイトル含有用

#                             figure_info = {"id": i+1, "file_name": file_name, "type": 0, "page_number": page_number, "itemId": item_id, "polygon": polygon}

#                             result_list.append(figure_info)

#                             count += 1

#                             if j+1 == len(boundingRegions):
#                                 subcount = 1

#         return result_list
    
#     # 表取得
#     def get_table(self, result, file_name):
#         result_list = []
#         page_number = 0
#         captionBR = []
#         polygon = []
#         total = 0
#         item_id = ""
#         count = 1
#         subcount = 1

#         if result["tables"]:
#             total = len(result["tables"])
#             for i, tables in enumerate(result["tables"]):
#                 if 'caption' in tables:
#                     caption_boundingRegions = tables['caption'].get("boundingRegions")  
#                     if caption_boundingRegions:  
#                         for item in caption_boundingRegions:  
#                             captionBR = item.get('polygon')

#                 boundingRegions = tables.get("boundingRegions")
#                 if boundingRegions:

#                     # # タイトル含有用
#                     # min_x = float('inf')  
#                     # min_y = float('inf')  
#                     # max_x = float('-inf')  
#                     # max_y = float('-inf')  
#                     # # タイトル含有用

#                     for j, item in enumerate(boundingRegions):
#                         if captionBR != item.get('polygon'):
#                             page_number = item.get('pageNumber')
#                             polygon = item.get('polygon')

#                             # # タイトル含有用
#                             # # 各ポリゴンの座標をチェックして最小/最大値を更新  
#                             # for k in range(0, len(polygon), 2):  
#                             #     x = polygon[k]  
#                             #     y = polygon[k+1]  
#                             #     if x < min_x:  
#                             #         min_x = x  
#                             #     if y < min_y:  
#                             #         min_y = y  
#                             #     if x > max_x:  
#                             #         max_x = x  
#                             #     if y > max_y:  
#                             #         max_y = y   
        
#                             # # 最小値と最大値から新しいバウンディングボックスを作成  
#                             # combined_polygon = [  
#                             #     min_x, min_y,  
#                             #     max_x, min_y,  
#                             #     max_x, max_y,  
#                             #     min_x, max_y  
#                             # ]
#                             # # タイトル含有用

#                             item_id = f"{i+1}-{subcount}"
#                             subcount += 1
#                             item_id = f"{i+1}"
                
#                         # # タイトル含有用
#                         # table_info = {"id": i+1, "file_name": file_name, "type": 1, "page_number": page_number, "itemId": item_id, "polygon": combined_polygon}
#                         # # タイトル含有用

#                             table_info = {"id": i+1, "file_name": file_name, "type": 1, "page_number": page_number, "itemId": item_id, "polygon": polygon}

#                             result_list.append(table_info)

#                             count += 1
                        
#                             if j+1 == len(boundingRegions):
#                                 subcount = 1

#         return result_list

#     def caption_prep(self, input_file_path, split_file_dir, chunk_size, file_name):
#         result = (self.get_ocr_result(input_file_path))["analyzeResult"]

#         # 座標で順番を整形できるか確認

#         figure_result = self.get_figure(result, file_name)
#         table_result = self.get_table(result, file_name)

#         return figure_result + table_result


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-1896103647814682>, line 2
      1 import pprint
----> 2 di = DocumentIntelligenceClient(endpoint="https://20240525test.cognitiveservices.azure.com/",key="242dc7331a484ecdae1f15ec90c957a8",
      3                                 api_version="2024-07-31-preview")
      5 di.custom_flag = False
      7 filepath = "/dbfs/mnt/rag-poc-q2-container/pdfs/ノウハウ_機密事項あり取り扱い注意/ゴム/NRU007_ゴムの加工.pdf"

NameError: name 'DocumentIntelligenceClient' is not defined